# Notebook 22d — Explainability: Can You Tell the Customer Why?
**AI Models and Agents for Business · Week 11 · Session 22**

Rudin's paper argues something stronger than "black boxes are risky." She argues that for high-stakes decisions, an interpretable model is usually **just as accurate** — so accepting a black box buys you nothing and costs you accountability.

This notebook tests that claim on a loan approval dataset, then builds the artifact a real lender is legally required to produce.

**No API key needed.**

| Part | You'll do |
|---|---|
| A | Train an interpretable model and a black box on the same data — compare accuracy |
| B | Read *why* the interpretable model decided what it decided |
| C | Try to explain the black box after the fact, and see what you lose |
| D | Generate an adverse action notice — the legally required "here's why you were denied" |

**Why this is a business problem, not a technical one:** under the U.S. Equal Credit Opportunity Act (Regulation B), a lender that denies credit must either list the *specific principal reasons* in the denial notice, or tell the applicant they can get those reasons by asking within 60 days (the lender then has 30 days to provide them). Either way, the lender must be able to produce the reasons, and they must be the factors that actually drove the decision. "Our model said no" is not a lawful answer. Explainability here is a compliance requirement, not a nice-to-have.

## Cell 1 — Setup and a loan applicant dataset

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

rng = np.random.RandomState(11)
n = 1200

df = pd.DataFrame({
    "credit_score":     rng.normal(680, 70, n).clip(450, 850).round(0),
    "debt_to_income":   rng.uniform(0.05, 0.65, n).round(3),
    "years_employed":   rng.exponential(5, n).clip(0, 35).round(1),
    "loan_to_income":   rng.uniform(0.5, 6.0, n).round(2),
    "late_payments_2y": rng.poisson(0.8, n),
})

# True repayment logic (what actually drives default), plus irreducible randomness
score = (
    (df.credit_score - 680) / 70 * 1.5
    - (df.debt_to_income - 0.35) / 0.15 * 1.1
    + np.minimum(df.years_employed, 10) / 10 * 0.6
    - (df.loan_to_income - 3.0) / 1.5 * 0.9
    - df.late_payments_2y * 0.55
    + rng.normal(0, 1.0, n)
)
df["repaid"] = (score > -0.3).astype(int)

FEATURES = ["credit_score", "debt_to_income", "years_employed", "loan_to_income", "late_payments_2y"]
X, y = df[FEATURES], df["repaid"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=11, stratify=y)

print(f"{n} applicants | {y.mean():.1%} repaid")
print(f"Train: {len(X_train)}  Test: {len(X_test)}")
df.head()

---
## Part A: Interpretable vs. black box — is there actually a tradeoff?

**Model 1 (interpretable):** logistic regression. Five numbers you can read and audit.
**Model 2 (black box):** random forest with 300 trees. Genuinely accurate, effectively unreadable.

In [ ]:
glass_box = make_pipeline(StandardScaler(), LogisticRegression())
glass_box.fit(X_train, y_train)

black_box = RandomForestClassifier(n_estimators=300, random_state=11)
black_box.fit(X_train, y_train)

rows = []
for name, model in [("Logistic regression (interpretable)", glass_box),
                    ("Random forest, 300 trees (black box)", black_box)]:
    acc = accuracy_score(y_test, model.predict(X_test))
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    rows.append((name, acc, auc))

print(f"{'Model':<40}{'Accuracy':<12}{'ROC-AUC'}")
print("-" * 62)
for name, acc, auc in rows:
    print(f"{name:<40}{acc:<12.1%}{auc:.3f}")

gap_acc = rows[1][1] - rows[0][1]
gap_auc = rows[1][2] - rows[0][2]
print()
print(f"Black box advantage: {gap_acc:+.1%} accuracy, {gap_auc:+.3f} AUC")
print()
print("How many parameters would you have to read to audit each model?")
print(f"  Logistic regression : {len(FEATURES)} coefficients + 1 intercept")
print(f"  Random forest       : 300 trees, {sum(t.tree_.node_count for t in black_box.estimators_):,} decision nodes total")

### 💬 Discussion — Part A
- What did the black box actually buy you here, in accuracy terms?
- Rudin's argument is that on structured, meaningful features like these, the accuracy gap is usually small or nonexistent — and that people accept black boxes out of habit rather than necessity. Does this result support her?
- When *would* you expect a black box to win by a lot? (Hint: think about images, audio, and raw text — none of which look like this table.)
- **Look at Cell 1:** repayment here depends on the five factors in a mostly straight-line way, which is exactly the shape logistic regression assumes. Does that stack the deck in the interpretable model's favor? Would you expect the same tie if the true relationship had sharp thresholds and interactions?
- **Careful:** this is one synthetic dataset. Rudin's claim is that the gap is often small on real, structured, high-stakes data, not that interpretable models always match black boxes. What would you need to run before generalizing from it?

---
## Part B: Reading the interpretable model

A logistic regression's coefficients are the explanation. Positive means "pushes toward approval," negative means "pushes toward denial." Because we standardized the inputs, the magnitudes are directly comparable.

In [ ]:
coefs = glass_box.named_steps["logisticregression"].coef_[0]
coef_table = (pd.DataFrame({"feature": FEATURES, "coefficient": coefs.round(3)})
              .assign(direction=lambda d: np.where(d.coefficient > 0, "→ approval", "→ denial"),
                      strength=lambda d: d.coefficient.abs().round(3))
              .sort_values("strength", ascending=False)
              .reset_index(drop=True))
print(coef_table.to_string(index=False))

plt.figure(figsize=(8, 4))
order = coef_table.sort_values("coefficient")
colors = ["firebrick" if v < 0 else "steelblue" for v in order.coefficient]
plt.barh(order.feature, order.coefficient, color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Coefficient (standardized) — negative pushes toward denial")
plt.title("The entire model, in one chart")
plt.tight_layout(); plt.show()

### 💬 Discussion — Part B
- Which factor carries the most weight? Is that what a loan officer would expect?
- Every one of these five factors is defensible to a regulator. Now imagine one of them were `zip_code` or `first_name_frequency`. Would the model still be *accurate*? Would it still be *lawful*? (Direct callback to Session 5's Amazon proxy-bias case.)
- This chart **is** the model documentation. What would the equivalent artifact look like for 300 trees?

---
## Part C: Explaining the black box after the fact

You can't read a random forest, so you probe it instead: shuffle one feature, see how much accuracy drops, and call that the feature's importance. This is **permutation importance** — a post-hoc explanation.

Watch what it gives you, and what it doesn't.

In [ ]:
perm = permutation_importance(black_box, X_test, y_test, n_repeats=20, random_state=11)

pi = (pd.DataFrame({"feature": FEATURES,
                    "importance": perm.importances_mean.round(4),
                    "std": perm.importances_std.round(4)})
      .sort_values("importance", ascending=False).reset_index(drop=True))
print("Black-box permutation importance:")
print(pi.to_string(index=False))

print()
print("Compare rankings:")
gb_rank = list(coef_table.feature)
bb_rank = list(pi.feature)
print(f"{'Rank':<6}{'Interpretable model':<22}{'Black box (post-hoc)'}")
print("-" * 54)
n_agree = 0
for i, (a, b) in enumerate(zip(gb_rank, bb_rank), 1):
    same = (a == b)
    n_agree += same
    flag = "" if same else "   <-- rankings differ"
    print(f"{i:<6}{a:<22}{b}{flag}")

print()
print(f"The two methods agree on {n_agree} of {len(FEATURES)} ranks.")
print("Global agreement is reassuring -- but see the discussion below for what")
print("it still does NOT give you.")

### 💬 Discussion — Part C
- Permutation importance tells you which features matter **on average, across everyone**. Notice what it does *not* tell you: whether high debt-to-income pushes a *specific* applicant toward approval or denial, or by how much.
- If a denied applicant calls and asks "what do I need to change to get approved next time?", can you answer from the permutation importance table? Can you answer from Part B's coefficients?
- On this dataset the two rankings come out **identical**, which is genuinely reassuring: the post-hoc explanation isn't inventing a false story about which factors matter globally. But notice how little that buys you — agreeing on a global ranking is not the same as being able to justify one person's denial.
- This is Rudin's deeper objection: a post-hoc explanation is a *story about* the model, not the model itself. Nothing guarantees the story stays faithful — on different data, or with a different explanation method, it can drift, and there's no ground truth to check it against. Why is that a serious problem in regulated lending?

---
## Part D: The adverse action notice

This is the deliverable that makes explainability concrete. Below, we take a single denied applicant and generate the specific principal reasons for denial — computed directly from the interpretable model's own arithmetic, not guessed at afterward.

Regulation B lets the lender choose between listing the reasons in the notice and telling the applicant how to request them. Here we list them. Either way, the reasons must be the ones that actually drove *this* decision. With an interpretable model, that is the model's own arithmetic. With a black box, you would be reporting whatever a post-hoc method ranked highest, which may not be the same thing.

For each factor we compute its **contribution** = standardized value × coefficient. The most negative contributions are the reasons for denial.

In [ ]:
scaler = glass_box.named_steps["standardscaler"]
lr = glass_box.named_steps["logisticregression"]

def explain_decision(applicant_row, threshold=0.5):
    x = applicant_row[FEATURES].to_frame().T.astype(float)
    x_scaled = scaler.transform(x)[0]
    contributions = x_scaled * lr.coef_[0]
    prob = float(lr.predict_proba(scaler.transform(x))[0][1])
    detail = (pd.DataFrame({
        "factor": FEATURES,
        "applicant_value": applicant_row[FEATURES].values,
        "contribution": contributions.round(3),
    }).sort_values("contribution").reset_index(drop=True))
    return prob, detail

READABLE = {
    "credit_score":     "Credit score",
    "debt_to_income":   "Debt-to-income ratio",
    "years_employed":   "Length of employment",
    "loan_to_income":   "Requested loan size relative to income",
    "late_payments_2y": "Recent late payments (24 months)",
}

# Find a denied applicant in the test set
probs = glass_box.predict_proba(X_test)[:, 1]
denied_idx = X_test.index[np.argsort(probs)[120]]  # clearly denied, but not an absurd outlier
applicant = df.loc[denied_idx]

prob, detail = explain_decision(applicant)
reasons = detail[detail.contribution < 0].head(3)

print("=" * 64)
print("NOTICE OF ADVERSE ACTION — CREDIT APPLICATION")
print("=" * 64)
print(f"Applicant reference : #{denied_idx}")
print(f"Decision            : DENIED")
print(f"Model repayment probability: {prob:.1%} (approval threshold 50.0%)")
print()
print("Principal reasons for this decision, most significant first:")
for i, r in enumerate(reasons.itertuples(), 1):
    print(f"  {i}. {READABLE[r.factor]} — your value: {r.applicant_value}")
print()
print("Factors that counted in your favor:")
favorable = detail[detail.contribution > 0]
if len(favorable):
    for r in favorable.sort_values('contribution', ascending=False).itertuples():
        print(f"  • {READABLE[r.factor]} — your value: {r.applicant_value}")
else:
    print("  (none)")
print()
print("Questions about these reasons? Contact the lender's credit department.")
# A real notice must also include the ECOA anti-discrimination statement and the name and
# address of the lender's federal regulator (Regulation B, 12 CFR 1002.9).
print("=" * 64)
print()
print("Full contribution breakdown for this applicant:")
print(detail.to_string(index=False))

## Cell 7 — ✏️ CHANGE ME: explain a different applicant

Pick any applicant and generate their notice. Try a borderline case (probability near 50%) — those are the ones that generate complaints and lawsuits.

In [ ]:
# ✏️ CHANGE ME — try rank 0 (most denied), or find a borderline case
RANK = 120        # 0 = most denied. Try 0 (extreme), 120 (typical), 273 (borderline)
order = np.argsort(probs)
idx = X_test.index[order[RANK]]

# For a borderline applicant instead, uncomment:
# idx = X_test.index[int(np.argmin(np.abs(probs - 0.5)))]

person = df.loc[idx]
p, d = explain_decision(person)
print(f"Applicant #{idx} | repayment probability {p:.1%} -> {'APPROVED' if p >= 0.5 else 'DENIED'}\n")
print(d.to_string(index=False))
print()
top_neg = d.iloc[0]
print(f"Single biggest reason against: {READABLE[top_neg.factor]} (value {top_neg.applicant_value})")

### 💬 Discussion — Part D
- You just produced a specific, per-person, arithmetically-traceable reason for denial. Could you have produced this from the random forest? What would you have had to bolt on?
- A borderline applicant at 49.8% is denied and a 50.1% applicant is approved. Both get a notice listing the same "principal reasons." Is that honest? What's actually driving the different outcomes?
- **Alignment vs. capability (today's core distinction):** this model is *capable* — it predicts repayment well. Suppose it were *misaligned*: accurate at predicting repayment, but achieving that accuracy partly through a proxy for race. Which of the four parts of this notebook would have caught that, and which would have missed it?
- **Model risk framing:** you're on the model risk committee reviewing this for production approval. List three things you'd require beyond what this notebook produced.

## ✅ Notebook 22d complete — Reflection questions

These prep you for **Quiz 6** (Session 25).

1. State Rudin's central claim in one sentence, and name the piece of evidence in Part A that speaks to it.
2. What is the difference between an *interpretable* model and an *explained* model?
3. Permutation importance gave a global ranking. Name one question it cannot answer that Part D's contribution breakdown can.
4. U.S. credit law requires a lender to be able to state the real reasons for a denial. Why does that make explainability a compliance requirement, and why is it easier to meet with an interpretable model than with a black box plus a post-hoc explanation?
5. Give one business context where you'd accept a black box, and justify why the accountability cost is acceptable there.

---
*Next session: the AGI debate. Today's alignment-vs-capability distinction is one of the central fault lines — bring it with you.*